# 🚀 LoRA Compress - Autoresearch on Google Colab

This notebook runs the LoRA compression autoresearch on Colab's GPU (T4) with results stored in Google Drive.

**Benefits vs local CPU:**
- ~50-100× faster (T4 GPU vs Zen 5 CPU)
- Persistent results in Google Drive
- Can resume interrupted runs

## Setup Instructions:
1. **Runtime → Change runtime type → Select T4 GPU**
2. Run cells in order
3. First run will ask for Google Drive permission

## Step 1: Mount Google Drive

All databases and results will be stored here for persistence.

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Create project directory in Drive
DRIVE_BASE = '/content/drive/MyDrive/LoRA_Compress'
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/databases', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/compressed_models', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/results', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/checkpoints', exist_ok=True)

print(f"✅ Drive mounted at: {DRIVE_BASE}")
print(f"📁 Databases: {DRIVE_BASE}/databases")
print(f"📁 Compressed models: {DRIVE_BASE}/compressed_models")
print(f"📁 Results: {DRIVE_BASE}/results")

## Step 2: Clone Repository

In [ ]:
%cd /content
!git clone https://github.com/qades/loracompress.git
%cd loracompress
!pwd

## Step 3: Install Dependencies

In [ ]:
# Install required packages
!pip install -q transformers torch optuna tqdm

# Verify GPU is available
import torch
print(f"\n🔥 PyTorch version: {torch.__version__}")
print(f"🔥 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")
    print(f"🔥 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 4: Autoresearch Configuration

Choose your target quality and configure settings:

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION - Modify these values
# ═══════════════════════════════════════════════════════════════

# Target L1 error percentage
# 5.0 = easier target, faster convergence
# 3.0 = stricter target, Q4_K_M benchmark quality
TARGET_QUALITY = 3.0

# Number of trials (Optuna optimization runs)
# 50 = good balance of speed/quality
# 100 = more thorough search, takes longer
N_TRIALS = 50

# Layer shape to optimize (default: 576x576 for SmolLM2-135M q_proj)
# You can change this for different layer dimensions
LAYER_SHAPE = (576, 576)

# Advanced mode (trap detection + adaptive noise)
# True = more robust but slightly slower
# False = simpler, faster
ADVANCED_MODE = True

# Resume from previous run if database exists?
RESUME = True

# ═══════════════════════════════════════════════════════════════

# Derived paths (stored in Drive)
DB_PATH = f'{DRIVE_BASE}/databases'
RESULTS_PATH = f'{DRIVE_BASE}/results'

# Study name based on target
STUDY_NAME = f"l1_quality_{TARGET_QUALITY}pct"
DB_FILE = f"{DB_PATH}/optuna_{STUDY_NAME}.db"
OUTPUT_FILE = f"{RESULTS_PATH}/autoresearch_l1_{TARGET_QUALITY}pct.json"

print("⚙️ Configuration:")
print(f"  Target quality: <{TARGET_QUALITY}% L1 error")
print(f"  Trials: {N_TRIALS}")
print(f"  Layer shape: {LAYER_SHAPE}")
print(f"  Advanced mode: {ADVANCED_MODE}")
print(f"  Resume: {RESUME}")
print(f"\n📂 Drive storage:")
print(f"  Database: {DB_FILE}")
print(f"  Results: {OUTPUT_FILE}")

## Step 5: Run Autoresearch

This will take 10-30 minutes depending on configuration. Progress is saved to Drive after each trial.

In [ ]:
import sys
sys.path.insert(0, '/content/loracompress/scripts')

import torch
import optuna
import json
import time
import os
from transformers import AutoModelForCausalLM
from autoresearch_l1_quality import (
    train_lora_layer_advanced,
    compute_l1_error,
    objective
)

# Set device to GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}\n")

# ═══════════════════════════════════════════════════════════════
# Load target weight
# ═══════════════════════════════════════════════════════════════
print("Loading model to extract sample weight...")
model = AutoModelForCausalLM.from_pretrained(
    'HuggingFaceTB/SmolLM2-135M', 
    trust_remote_code=True
)

target_weight = None
for name, param in model.named_parameters():
    if 'q_proj' in name and param.shape == LAYER_SHAPE:
        target_weight = param.data
        print(f"✅ Using real weight: {name} {param.shape}")
        break

if target_weight is None:
    print(f"⚠️ No layer with shape {LAYER_SHAPE} found, using synthetic")
    torch.manual_seed(42)
    target_weight = torch.randn(LAYER_SHAPE) * 0.1

# Move to GPU
target_weight = target_weight.to(device)

print("\n" + "="*70)
print("🚀 Starting L1 Quality Autoresearch")
print("="*70)
print(f"Target quality: <{TARGET_QUALITY}% L1 error")
print(f"Benchmark: Q4_K_M = 3.4% error, 70-75% size reduction")
print(f"Device: {device.upper()}")
print("="*70 + "\n")

# ═══════════════════════════════════════════════════════════════
# Setup Optuna study with Drive storage
# ═══════════════════════════════════════════════════════════════
storage_url = f"sqlite:///{DB_FILE}"

# Check if resuming
if os.path.exists(DB_FILE) and RESUME:
    print(f"📂 Found existing database, resuming study...")
    load_existing = True
else:
    print(f"🆕 Starting fresh study...")
    load_existing = False
    # Ensure directory exists
    os.makedirs(os.path.dirname(DB_FILE), exist_ok=True)

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=storage_url,
    direction='minimize',
    load_if_exists=RESUME
)

# ═══════════════════════════════════════════════════════════════
# Run optimization
# ═══════════════════════════════════════════════════════════════
def wrapped_objective(trial):
    return objective(
        trial, 
        target_weight, 
        TARGET_QUALITY, 
        noise_std=0.0, 
        noise_every=50, 
        advanced_mode=ADVANCED_MODE
    )

print(f"\nRunning {N_TRIALS} trials...\n")
start_time = time.time()

# Progress callback
def progress_callback(study, trial):
    if trial.number % 5 == 0:
        elapsed = time.time() - start_time
        print(f"Trial {trial.number}/{N_TRIALS} | "
              f"Best score: {study.best_value:.1f} | "
              f"Elapsed: {elapsed/60:.1f}min")

study.optimize(
    wrapped_objective, 
    n_trials=N_TRIALS, 
    show_progress_bar=True,
    callbacks=[progress_callback]
)

total_time = time.time() - start_time
print(f"\n✅ Completed {N_TRIALS} trials in {total_time/60:.1f} minutes")
print(f"   Average: {total_time/N_TRIALS:.1f}s per trial")

## Step 6: Display Results

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Collect and display results
# ═══════════════════════════════════════════════════════════════
results = []
for trial in study.trials:
    if trial.state == optuna.trial.TrialState.COMPLETE:
        result = {
            'rank': trial.params.get('rank'),
            'lr': trial.params.get('lr'),
            'epochs': trial.params.get('epochs'),
            'error': trial.user_attrs.get('error'),
            'compression': trial.user_attrs.get('compression'),
            'size_reduction': trial.user_attrs.get('size_reduction'),
            'actual_epochs': trial.user_attrs.get('actual_epochs'),
            'score': trial.value,
            'scheduler': trial.user_attrs.get('scheduler'),
        }
        if 'noise_mode' in trial.user_attrs:
            result['noise_mode'] = trial.user_attrs.get('noise_mode')
            result['adaptive_noise'] = trial.user_attrs.get('adaptive_noise')
            result['detect_traps'] = trial.user_attrs.get('detect_traps')
        results.append(result)

# Sort by error
results.sort(key=lambda x: x['error'])

# Display top results
print("\n" + "="*80)
print("🏆 TOP 10 BY QUALITY (lowest L1 error)")
print("="*80)
print(f"{'Rank':>6} {'LR':>10} {'Epochs':>8} {'Error%':>10} {'Compr':>10} {'Score':>10}")
print("-"*80)

for r in results[:10]:
    marker = "✓" if r['error'] <= TARGET_QUALITY else "✗"
    print(f"{r['rank']:>6} {r['lr']:>10.4f} {r['actual_epochs']:>8} "
          f"{r['error']:>9.2f}{marker} {r['compression']:>9.1f}x {r['score']:>10.1f}")

# Find best configs under different thresholds
print("\n" + "="*80)
print("📋 RECOMMENDED CONFIGURATIONS")
print("="*80)

for threshold in [3.0, 4.0, 5.0]:
    good = [r for r in results if r['error'] <= threshold]
    if good:
        best_compression = max(good, key=lambda x: x['compression'])
        print(f"\nFor <{threshold}% error (BEST COMPRESSION):")
        print(f"  Rank={best_compression['rank']}, LR={best_compression['lr']:.4f}, "
              f"Epochs={best_compression['epochs']}")
        print(f"  → Error={best_compression['error']:.2f}%, "
              f"Compression={best_compression['compression']:.1f}x")
        
        # Most efficient
        most_efficient = min(good, key=lambda x: x['rank'] * x['actual_epochs'])
        print(f"For <{threshold}% error (MOST EFFICIENT):")
        print(f"  Rank={most_efficient['rank']}, LR={most_efficient['lr']:.4f}, "
              f"Epochs={most_efficient['epochs']}")
        print(f"  → Error={most_efficient['error']:.2f}%, "
              f"Effort={most_efficient['rank'] * most_efficient['actual_epochs']/1e6:.1f}M")

## Step 7: Save Results to Drive

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Save results to Drive
# ═══════════════════════════════════════════════════════════════
output = {
    'target_shape': LAYER_SHAPE,
    'target_quality': TARGET_QUALITY,
    'n_trials': N_TRIALS,
    'device': device,
    'total_time_seconds': total_time,
    'best_config': {
        'rank': study.best_params.get('rank'),
        'lr': study.best_params.get('lr'),
        'epochs': study.best_params.get('epochs'),
        'error': study.best_trial.user_attrs.get('error'),
        'compression': study.best_trial.user_attrs.get('compression'),
    },
    'all_results': results,
}

with open(OUTPUT_FILE, 'w') as f:
    json.dump(output, f, indent=2)

print(f"✅ Results saved to: {OUTPUT_FILE}")
print(f"\n📊 Summary:")
print(f"  Best L1 error: {output['best_config']['error']:.2f}%")
print(f"  Best compression: {output['best_config']['compression']:.1f}x")
print(f"  Optimal rank: {output['best_config']['rank']}")
print(f"  Optimal LR: {output['best_config']['lr']:.4f}")
print(f"  Optimal epochs: {output['best_config']['epochs']}")

## 🔄 Bonus: Resume or Run Different Targets

After running once, you can:
1. **Resume** the same study by just re-running Step 5 (set `RESUME = True`)
2. **Run a different target** by changing `TARGET_QUALITY` and re-running from Step 4
3. **Download results** from your Drive under `MyDrive/LoRA_Compress/`

## 📊 Speed Comparison

| Platform | Time per Trial | 50 Trials Estimate |
|----------|---------------|-------------------|
| Strix Halo CPU (your local) | ~360s | ~5 hours |
| Colab T4 GPU | ~5-15s | **~10-20 minutes** |
| Colab CPU | ~100s | ~1.5 hours |

**You're getting 20-30× speedup with Colab T4!** 🚀